In [46]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor, export_text
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import xgboost as xgb

In [36]:
#! pip install xgboost

^C


In [21]:
data = 'https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv'

In [22]:
df=pd.read_csv(data)
df.to_csv('car_fuel_efficiency_2026.csv', index=False)
df.columns = df.columns.str.lower()
df


,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [23]:
df = df.fillna(0)
df

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,0.0,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,0.0,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0
...,...,...,...,...,...,...,...,...,...,...,...
9995,2005,USA,Gasoline,All-wheel drive,4,2470,6,271.0,4690,18.9,27.4
9996,1993,Asia,Gasoline,All-wheel drive,2,2300,6,244.0,4370,19.0,30.0
9997,2014,USA,Gasoline,All-wheel drive,3,2480,6,267.0,4590,18.4,28.1
9998,2004,USA,Hybrid,Rear-wheel drive,5,2180,6,270.0,4450,20.9,32.2


In [26]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)


df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [27]:
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [ ]:
features = list(df_train.columns)

train_dicts = df_train[features].to_dict(orient='records')
val_dicts = df_val[features].to_dict(orient='records')

dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

print(f"Train shape: {X_train.shape}, Val shape: {X_val.shape}")

Train shape: (6000, 16), Val shape: (2000, 16)


Question 1


In [ ]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

tree_rules = export_text(dt, feature_names=list(dv.get_feature_names_out()))
print(tree_rules)

split_feature_index = np.argmax(dt.feature_importances_)
split_feature = dv.get_feature_names_out()[split_feature_index]
print(f"Split feature in DictVectorizer: {split_feature}")

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]

Split feature in DictVectorizer: model_year


Question 2

In [30]:
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)
rmse = root_mean_squared_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.4f}")

Validation RMSE: 1.8332


Question 3


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

estimators_list = [10, 50, 100, 150]
results = {}

for n in estimators_list:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    
    y_pred = rf.predict(X_val)
    val_rmse = root_mean_squared_error(y_val, y_pred)
    
    results[n] = round(val_rmse, 3)
    print(f"n_estimators = {n:<4} -> RMSE: {val_rmse:.4f} | Rounded (3 decimals): {results[n]:.3f}")

best_n = min(results, key=results.get)
print("-" * 45)
print(f"The best n_estimators is: {best_n} (with RMSE = {results[best_n]})")

n_estimators = 10   -> RMSE: 1.8332 | Rounded (3 decimals): 1.833
n_estimators = 50   -> RMSE: 1.7740 | Rounded (3 decimals): 1.774
n_estimators = 100  -> RMSE: 1.7698 | Rounded (3 decimals): 1.770
n_estimators = 150  -> RMSE: 1.7692 | Rounded (3 decimals): 1.769
---------------------------------------------
The best n_estimators is: 150 (with RMSE = 1.769)


Question 4


In [42]:
import numpy as np

depths = [10, 15, 20, 25]
n_estimators_list = [10, 50, 100, 150]

mean_scores = {}

for depth in depths:
    rmses = []
    for n in n_estimators_list:
        rf = RandomForestRegressor(
            max_depth=depth,
            n_estimators=n,
            random_state=1,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)
        
        y_pred = rf.predict(X_val)
        rmse = root_mean_squared_error(y_val, y_pred)
        rmses.append(rmse)
        
    mean_rmse = np.mean(rmses)
    mean_scores[depth] = mean_rmse
    print(f"max_depth = {depth:<2} | Individual RMSEs: {[round(r, 4) for r in rmses]}")
    print(f"             | Mean RMSE: {mean_rmse:.4f}\n")

best_depth = min(mean_scores, key=mean_scores.get)
print("-" * 40)
print(f"Best max_depth by lowest mean RMSE: {best_depth}")

max_depth = 10 | Individual RMSEs: [1.7823, 1.7475, 1.744, 1.7437]
             | Mean RMSE: 1.7544

max_depth = 15 | Individual RMSEs: [1.8314, 1.7709, 1.7679, 1.7668]
             | Mean RMSE: 1.7842

max_depth = 20 | Individual RMSEs: [1.8373, 1.7724, 1.7689, 1.7685]
             | Mean RMSE: 1.7868

max_depth = 25 | Individual RMSEs: [1.8369, 1.7721, 1.7683, 1.7679]
             | Mean RMSE: 1.7863

----------------------------------------
Best max_depth by lowest mean RMSE: 10


Question 5


In [44]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer

# 1. Train the model with the exact parameters
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)
rf.fit(X_train, y_train)

# 2. Get feature names and their importance scores
feature_names = dv.get_feature_names_out()
importances = rf.feature_importances_

# 3. Sum importance per original column (mapping back any one-hot prefixes like origin=USA)
feature_imp_dict = {}
for name, imp in zip(feature_names, importances):
    orig_name = name.split('=')[0]
    feature_imp_dict[orig_name] = feature_imp_dict.get(orig_name, 0.0) + imp

# 4. Compare the 4 requested candidates
candidates = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']

print("Feature Importances:")
for col in candidates:
    print(f"{col:<22}: {feature_imp_dict.get(col, 0.0):.4f}")

# 5. Identify the winner
most_important = max(candidates, key=lambda c: feature_imp_dict.get(c, 0.0))
print("-" * 35)
print(f"Most important feature: {most_important}")

Feature Importances:
vehicle_weight        : 0.2033
horsepower            : 0.0783
acceleration          : 0.0550
engine_displacement   : 0.0613
-----------------------------------
Most important feature: vehicle_weight


Question 6


In [48]:
import xgboost as xgb
import re

# 1. Clean feature names for XGBoost (XGBoost does not allow special characters like '[', ']', '<', or '=')
feature_names = dv.get_feature_names_out().tolist()
clean_feature_names = [re.sub(r'[\[\]<>=]', '_', f) for f in feature_names]

# 2. Create DMatrix for train and validation
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=clean_feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=clean_feature_names)

# 3. Create the watchlist
watchlist = [(dtrain, 'train'), (dval, 'val')]

# 4. Evaluate eta = 0.3 vs eta = 0.1 for 100 rounds
scores = {}

for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
        'objective': 'reg:squarederror',
        'nthread': 8,
        'seed': 1,
        'verbosity': 1,
    }
    
    evals_result = {}
    
    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        evals_result=evals_result,
        verbose_eval=False
    )
    
    # Extract the validation RMSE at the 100th boosting round (index -1)
    final_val_rmse = evals_result['val']['rmse'][-1]
    scores[eta] = final_val_rmse
    print(f"eta = {eta:<4} -> Final Validation RMSE (round 100): {final_val_rmse:.4f}")

# 5. Identify the better parameter
best_eta = min(scores, key=scores.get)
print("-" * 50)
print(f"Best eta based on validation RMSE: {best_eta}")

eta = 0.3  -> Final Validation RMSE (round 100): 1.8265
eta = 0.1  -> Final Validation RMSE (round 100): 1.7248
--------------------------------------------------
Best eta based on validation RMSE: 0.1
